# Proyek Akhir Teknik Kompilasi: Representasi Tahapan Kompilasi
**Nama:** Muhammad Mabi Palaka
**NIM:** 231011401691
**Prodi:** Teknik Informatika (Semester 6)  

---
## 1. Pilihan Konstruksi & Pattern (BNF)
Konstruksi yang dipilih untuk diimplementasikan adalah **Perulangan (While Loop)**.

### Aturan Sintaks (Backus-Naur Form):
```text
<while_stmt>  ::= "while" "(" <condition> ")" "{" <statement> "}"
<condition>   ::= <identifier> <rel_op> <value>
<statement>   ::= <identifier> "=" <identifier> <math_op> <value>
<rel_op>      ::= "<" | ">" | "<=" | ">=" | "==" | "!="
<math_op>     ::= "+" | "-" | "*" | "/"
```

In [1]:
import re

class WhileCompiler:
    def __init__(self, source_code):
        self.source_code = source_code
        self.label_counter = 1
        # Simulasi Tabel Simbol untuk Analisis Semantik
        self.symbol_table = {"counter": "int", "limit": "int", "total": "int"}

    def new_label(self):
        lbl = f"L{self.label_counter}"
        self.label_counter += 1
        return lbl

    def lexical_analysis(self):
        """
        Tahap Leksikal: Memecah kode sumber menjadi token-token spesifik.
        """
        token_specification = [
            ('WHILE',    r'while'),
            ('LPAREN',   r'\('),
            ('RPAREN',   r'\)'),
            ('LBRACE',   r'\{'),
            ('RBRACE',   r'\}'),
            ('OP',       r'[+\-*/=<>]'),
            ('ID',       r'[a-zA-Z_][a-zA-Z0-9_]*'),
            ('NUM',      r'\d+'),
            ('SKIP',     r'[ \t\n]+'), 
        ]
        
        tok_regex = '|'.join(f'(?P<{name}>{pattern})' for name, pattern in token_specification)
        tokens = []
        
        for mo in re.finditer(tok_regex, self.source_code):
            kind = mo.lastgroup
            value = mo.group()
            if kind == 'SKIP':
                continue
            tokens.append((kind, value))
        return tokens

    def syntax_semantic_analysis(self, tokens):
        """
        Tahap Sintaksis & Semantik: Validasi struktur aturan dan token.
        """
        try:
            token_kinds = [t[0] for t in tokens]
            
            if token_kinds[0] != 'WHILE':
                raise SyntaxError("Sintaks Error: Harus dimulai dengan 'while'")
                
            lparen_idx = token_kinds.index('LPAREN')
            rparen_idx = token_kinds.index('RPAREN')
            condition_tokens = tokens[lparen_idx+1 : rparen_idx]
            
            lbrace_idx = token_kinds.index('LBRACE')
            rbrace_idx = token_kinds.index('RBRACE')
            body_tokens = tokens[lbrace_idx+1 : rbrace_idx]
            
        except (ValueError, IndexError):
            raise SyntaxError("Sintaks Error: Blok kurung () atau {} tidak lengkap.")
 
        for kind, value in tokens:
            if kind == 'ID':
                if value not in self.symbol_table:
                    raise NameError(f"Semantik Error: Variabel '{value}' belum dideklarasikan!")

        condition_str = " ".join([t[1] for t in condition_tokens])
        body_str = " ".join([t[1] for t in body_tokens])
        
        return condition_str, body_str

    def generate_tac(self):
        """
        Tahap Akhir: Menghasilkan kode tiga alamat (Intermediate Code / TAC).
        """
        tokens = self.lexical_analysis()
        condition, body = self.syntax_semantic_analysis(tokens)
 
        label_start = self.new_label()
        label_end = self.new_label()
 
        tac = []
        tac.append(f"{label_start}:")
        tac.append(f"ifFalse {condition} goto {label_end}")
        tac.append(f"    {body}")
        tac.append(f"goto {label_start}")
        tac.append(f"{label_end}:")
        
        return tokens, tac

In [2]:
# --- Uji Coba Eksekusi Compiler ---
source_program = "while ( counter < limit ) { total = total + 5 }"
compiler = WhileCompiler(source_program)

try:
    tokens, tac_result = compiler.generate_tac()
    
    print("==================================================")
    print(f"INPUT SOURCE CODE:\n  {source_program}")
    print("==================================================\n")
    
    print("1. TAHAP ANALISIS LEKSIKAL (Tokens):")
    for t in tokens:
        print(f"   {t}")
        
    print("\n2 & 3. TAHAP SINTAKSIS & SEMANTIK:")
    print("   Status: Sukses (Struktur Benar & Variabel Terdaftar)")
    
    print("\n4. GENERASI THREE-ADDRESS CODE (TAC):")
    for line in tac_result:
        print(f"   {line}")
    print("==================================================")
        
except Exception as e:
    print(f"Kompilasi Gagal: {e}")

INPUT SOURCE CODE:
  while ( counter < limit ) { total = total + 5 }

1. TAHAP ANALISIS LEKSIKAL (Tokens):
   ('WHILE', 'while')
   ('LPAREN', '(')
   ('ID', 'counter')
   ('OP', '<')
   ('ID', 'limit')
   ('RPAREN', ')')
   ('LBRACE', '{')
   ('ID', 'total')
   ('OP', '=')
   ('ID', 'total')
   ('OP', '+')
   ('NUM', '5')
   ('RBRACE', '}')

2 & 3. TAHAP SINTAKSIS & SEMANTIK:
   Status: Sukses (Struktur Benar & Variabel Terdaftar)

4. GENERASI THREE-ADDRESS CODE (TAC):
   L1:
   ifFalse counter < limit goto L2
       total = total + 5
   goto L1
   L2:


## 2. Penjelasan Singkat Alur Proses Kompilasi
1. **Lexical Analysis (Scanner)**: Membaca teks mentah program dan mengelompokkan karakter menjadi Token spesifik (seperti `ID` untuk variabel, `WHILE` untuk keyword).
2. **Syntax Analysis (Parser)**: Memastikan urutan token menuruti aturan tata bahasa perulangan `while` yang baku (misal: kondisi wajib dalam kurung).
3. **Semantic Analysis**: Memeriksa konteks logika dan validitas variabel terhadap *Symbol Table* untuk mencegah penggunaan variabel gaib/ilegal.
4. **Code Generation (TAC)**: Mentransformasi bentuk logika tinggi ke instruksi tingkat rendah berbasis *conditional jump* menggunakan label `L1` dan `L2`.